# O1-08 — Training-Space and Descriptor Alignment

## Objective

This notebook completes the remaining Objective 1 alignment tasks identified
in the Robustness-Aware Inference Pipeline architecture briefing.

Objective 1 owns the pre-inference gate in front of the existing Gamma neural
network.

The remaining checks addressed here are:

1. reproduce the RYA/paper unreliability-matrix descriptors exactly;
2. compare incoming NN inputs against the ranges represented in EXP002 training
   metadata;
3. compare P-matrix structure against matrices represented in the available
   RYA instance/training data;
4. verify that compatibility screening is applied to the same X/Y instance
   that is passed into the Gamma neural network;
5. integrate these diagnostics into the existing O1 routing output without
   changing the already validated compatibility classifier.

The existing 54-feature compatibility classifier and its provisional threshold
of 0.10 are not retrained in this notebook.

Architecture mapping
The architecture briefing assigns SIN-WEI responsibility for:

mathematical validation of the unreliability matrix;
comparison with structures represented in training;
Gamma compatibility classification;
recall-focused screening;
documenting failure characteristics;
intercepting unsuitable cases before neural-network inference.
The classifier, recall tuning, failure analysis and pre-NN routing were completed in O1-04 through O1-07.

This notebook focuses only on the remaining training-space and descriptor alignment items.

## 0. Environment and RYA asset setup

This notebook is designed to run independently from O1-07.

The required RYA assets, EXP002 metadata, unreliability matrices and
feature-construction functions are therefore reloaded explicitly before the
training-space validation analysis begins.

In [16]:
from pathlib import Path
from io import StringIO
import json

import numpy as np
import pandas as pd

In [18]:
BASE_DIR = Path(
    r"C:\AIML Y2 S2\Capstone 1\RYA"
)

MODEL_DIR = (
    BASE_DIR
    / "final_network_model"
    / "final_network_model"
)

CODE_DIR = (
    BASE_DIR
    / "final_code"
    / "final_code"
)

P_MATRIX_FILE = (
    BASE_DIR
    / "unreliability-matrices.json"
)

METADATA_FILE = (
    MODEL_DIR
    / "EXP002.json"
)

print("BASE_DIR exists:", BASE_DIR.exists())
print("MODEL_DIR exists:", MODEL_DIR.exists())
print("P matrix file exists:", P_MATRIX_FILE.exists())
print("Metadata file exists:", METADATA_FILE.exists())

BASE_DIR exists: True
MODEL_DIR exists: True
P matrix file exists: True
Metadata file exists: True


In [20]:
with open(
    P_MATRIX_FILE,
    "r"
) as f:
    P_matrices = json.load(f)

print(
    "Loaded P matrices:",
    len(P_matrices)
)

print(
    "ry9 available:",
    "ry9" in P_matrices
)

Loaded P matrices: 59
ry9 available: True


In [22]:
with open(
    METADATA_FILE,
    "r"
) as f:
    rya_metadata = json.load(f)

print(
    "EXP002 feature count:",
    len(
        rya_metadata[
            "feature_labels"
        ]
    )
)

print(
    "First 5 features:",
    rya_metadata[
        "feature_labels"
    ][:5]
)

EXP002 feature count: 40
First 5 features: ['X_1', 'X_2', 'X_3', 'X_4', 'Y_1']


In [24]:
nn_minmax = pd.read_json(
    StringIO(
        rya_metadata[
            "min_max_df"
        ]
    )
)

print(
    "Min/max table shape:",
    nn_minmax.shape
)

display(
    nn_minmax.head()
)

Min/max table shape: (42, 3)


,column,min,max
0,X_1,0.0,0.956522
1,X_2,0.0,0.977778
2,X_3,0.0,0.969697
3,X_4,0.0,0.944444
4,Y_1,0.0,0.976190


In [26]:
def rya_moment_calculator(
    distribution
):
    """
    Reproduce the RYA moment calculation for
    a four-category Likert distribution.
    """

    distribution = np.asarray(
        distribution,
        dtype=float
    )

    support = np.arange(
        1,
        5,
        dtype=float
    )

    mean = float(
        np.sum(
            support
            * distribution
        )
    )

    variance = float(
        np.sum(
            ((support - mean) ** 2)
            * distribution
        )
    )

    stdev = float(
        np.sqrt(
            variance
        )
    )

    if stdev == 0:
        skewness = np.nan
        kurtosis = np.nan

    else:
        skewness = float(
            np.sum(
                ((support - mean) ** 3)
                * distribution
            )
            / (stdev ** 3)
        )

        kurtosis = float(
            np.sum(
                ((support - mean) ** 4)
                * distribution
            )
            / (stdev ** 4)
        )

    return (
        mean,
        stdev,
        skewness,
        kurtosis
    )

In [28]:
def rya_asymm_calc(
    P_matrix
):
    """
    Reproduce the asymmetry calculation used
    by the supplied RYA portal implementation.
    """

    P = np.asarray(
        P_matrix,
        dtype=float
    )

    differences = []

    for i in range(4):
        for j in range(
            i + 1,
            4
        ):
            differences.append(
                (
                    P[i, j]
                    - P[j, i]
                ) ** 2
            )

    return float(
        np.mean(
            differences
        )
    )

In [30]:
def build_rya_nn_features(
    x,
    y,
    P_matrix
):
    """
    Build the exact 40-feature input expected
    by the supplied RYA EXP002 neural network.
    """

    x = np.asarray(
        x,
        dtype=float
    )

    y = np.asarray(
        y,
        dtype=float
    )

    P_matrix = np.asarray(
        P_matrix,
        dtype=float
    )

    n_X = float(
        x.sum()
    )

    n_Y = float(
        y.sum()
    )

    if n_X <= 0 or n_Y <= 0:
        raise ValueError(
            "Both groups must contain observations."
        )

    X = x / n_X
    Y = y / n_Y

    XmY = np.abs(
        X - Y
    )

    (
        mean_X,
        stdev_X,
        skewness_X,
        kurtosis_X
    ) = rya_moment_calculator(
        X
    )

    (
        mean_Y,
        stdev_Y,
        skewness_Y,
        kurtosis_Y
    ) = rya_moment_calculator(
        Y
    )

    euclid_dist_XY = float(
        np.round(
            np.linalg.norm(
                X - Y
            ),
            4
        )
    )

    features = {}

    for i in range(4):
        features[
            f"X_{i+1}"
        ] = float(
            X[i]
        )

    for i in range(4):
        features[
            f"Y_{i+1}"
        ] = float(
            Y[i]
        )

    for i in range(4):
        features[
            f"XmY_{i+1}"
        ] = float(
            XmY[i]
        )

    features.update({
        "mean_X":
            mean_X,
        "stdev_X":
            stdev_X,
        "skewness_X":
            skewness_X,
        "kurtosis_X":
            kurtosis_X,

        "mean_Y":
            mean_Y,
        "stdev_Y":
            stdev_Y,
        "skewness_Y":
            skewness_Y,
        "kurtosis_Y":
            kurtosis_Y,

        "euclid_dist_XY":
            euclid_dist_XY,

        "n_X":
            n_X,

        "n_Y":
            n_Y
    })

    # Preserve NumPy row-major order used by
    # the supplied RYA portal implementation.
    flat_P = (
        P_matrix
        .flatten()
    )

    for i in range(16):
        features[
            f"P_{i}"
        ] = float(
            flat_P[i]
        )

    features[
        "asymm"
    ] = rya_asymm_calc(
        P_matrix
    )

    feature_series = pd.Series(
        features,
        dtype=float
    )

    feature_series = (
        feature_series
        .reindex(
            rya_metadata[
                "feature_labels"
            ]
        )
    )

    return feature_series

In [32]:
required_names = [
    "P_matrices",
    "rya_metadata",
    "nn_minmax",
    "build_rya_nn_features",
]

for name in required_names:
    print(
        name,
        "->",
        "FOUND" if name in globals() else "MISSING"
    )

P_matrices -> FOUND
rya_metadata -> FOUND
nn_minmax -> FOUND
build_rya_nn_features -> FOUND


In [34]:
def rya_certainty(P_matrix):
    """
    RYA/paper certainty descriptor:
    sum of the diagonal probabilities.
    """
    P = np.asarray(P_matrix, dtype=float)

    if P.shape != (4, 4):
        raise ValueError("Expected a 4x4 P matrix.")

    return float(np.trace(P))


def rya_asymmetry(P_matrix):
    """
    RYA/paper asymmetry descriptor:
    mean squared difference across the six unique
    off-diagonal symmetric pairs.
    """
    P = np.asarray(P_matrix, dtype=float)

    if P.shape != (4, 4):
        raise ValueError("Expected a 4x4 P matrix.")

    squared_differences = []

    for i in range(4):
        for j in range(i + 1, 4):

            squared_differences.append(
                (P[i, j] - P[j, i]) ** 2
            )

    return float(
        np.mean(
            squared_differences
        )
    )

In [36]:
def detector_asymmetry(P_matrix):
    P = np.asarray(
        P_matrix,
        dtype=float
    )

    return float(
        np.mean(
            (P - P.T) ** 2
        )
    )

In [38]:
P_ry9 = np.asarray(
    P_matrices["ry9"],
    dtype=float
)

print(
    "RYA certainty:",
    rya_certainty(P_ry9)
)

print(
    "RYA asymmetry:",
    rya_asymmetry(P_ry9)
)

print(
    "Detector asymmetry:",
    detector_asymmetry(P_ry9)
)

RYA certainty: 2.42
RYA asymmetry: 0.030716666666666666
Detector asymmetry: 0.0230375


### Descriptor alignment finding

The RYA/paper matrix descriptors were reproduced successfully.

For the `ry9` matrix:

- certainty = 2.42;
- RYA asymmetry = 0.03072;
- detector asymmetry = 0.02304.

The asymmetry values differ because the existing detector and the RYA/paper
definition use different averaging conventions.

This does not invalidate the trained detector, but both definitions should be
reported explicitly so that the production-facing validation stage remains
traceable to the RYA methodology.

## 2. Neural-network training-range diagnostics

The EXP002 metadata provides feature-wise minimum and maximum values for the
40 inputs expected by the existing Gamma neural network.

These values do not provide a full statistical distribution of the training
space, but they do allow a transparent extrapolation check.

For each candidate case, this section reports:

- whether all NN input features fall inside the recorded EXP002 range;
- the number of out-of-range features;
- which features are below the recorded minimum;
- which features are above the recorded maximum;
- the size of each exceedance in normalised units.

This diagnostic is initially treated as a warning rather than an automatic
fallback rule because no validated routing threshold has yet been established
for min/max exceedance.

In [43]:
def check_nn_training_range(
    nn_features,
    minmax_df,
    feature_order
):
    """
    Compare the 40 raw NN input features with the
    EXP002 feature-wise training ranges.
    """

    records = []

    for feature_name in feature_order:

        value = float(
            nn_features[
                feature_name
            ]
        )

        row = minmax_df[
            minmax_df["column"]
            == feature_name
        ]

        if len(row) != 1:
            raise ValueError(
                f"No unique min/max row for {feature_name}"
            )

        fmin = float(
            row["min"].iloc[0]
        )

        fmax = float(
            row["max"].iloc[0]
        )

        below = value < fmin
        above = value > fmax

        if fmax > fmin:
            normalised_position = (
                value - fmin
            ) / (
                fmax - fmin
            )
        else:
            normalised_position = 0.0

        if below:
            exceedance = (
                fmin - value
            ) / (
                fmax - fmin
            ) if fmax > fmin else np.nan

        elif above:
            exceedance = (
                value - fmax
            ) / (
                fmax - fmin
            ) if fmax > fmin else np.nan

        else:
            exceedance = 0.0

        records.append({
            "feature": feature_name,
            "value": value,
            "training_min": fmin,
            "training_max": fmax,
            "below_min": below,
            "above_max": above,
            "out_of_range":
                bool(below or above),
            "normalised_position":
                normalised_position,
            "normalised_exceedance":
                float(exceedance)
        })

    detail_df = pd.DataFrame(
        records
    )

    out_df = detail_df[
        detail_df[
            "out_of_range"
        ]
    ].copy()

    summary = {
        "within_training_range":
            len(out_df) == 0,

        "out_of_range_feature_count":
            int(len(out_df)),

        "out_of_range_features":
            out_df[
                "feature"
            ].tolist(),

        "max_normalised_exceedance":
            (
                float(
                    out_df[
                        "normalised_exceedance"
                    ].max()
                )
                if len(out_df) > 0
                else 0.0
            )
    }

    return summary, detail_df

In [45]:
ry9_nn_features = build_rya_nn_features(
    x=np.array(
        [40, 52, 7, 14],
        dtype=float
    ),
    y=np.array(
        [9, 17, 54, 12],
        dtype=float
    ),
    P_matrix=P_ry9
)

ry9_nn_features = (
    ry9_nn_features
    .reindex(
        rya_metadata[
            "feature_labels"
        ]
    )
)

ry9_range_summary, ry9_range_detail = (
    check_nn_training_range(
        nn_features=ry9_nn_features,
        minmax_df=nn_minmax,
        feature_order=
            rya_metadata[
                "feature_labels"
            ]
    )
)

ry9_range_summary

{'within_training_range': True,
 'out_of_range_feature_count': 0,
 'out_of_range_features': [],
 'max_normalised_exceedance': 0.0}

In [47]:
display(
    ry9_range_detail[
        ry9_range_detail[
            "out_of_range"
        ]
    ]
)

,feature,value,training_min,training_max,below_min,above_max,out_of_range,normalised_position,normalised_exceedance


### EXP002 range-check result

The `ry9` reference case falls completely within the feature-wise ranges
recorded in the EXP002 neural-network metadata.

- 40 NN input features checked;
- 0 features below the recorded training minimum;
- 0 features above the recorded training maximum;
- maximum normalised exceedance = 0.

This confirms that the range diagnostic behaves as expected for a known
RYA example.

The min/max check is used as an extrapolation diagnostic only. It is not
treated as a complete statistical measure of distance from the NN training
distribution and does not automatically trigger fallback.

In [50]:
SCREENING_DATA_FILE = (
    BASE_DIR
    / "O1_classifier_inputs_unlabelled.csv"
)

screening_data = pd.read_csv(
    SCREENING_DATA_FILE
)

print(
    "Screening data shape:",
    screening_data.shape
)

print(
    "Unique questions:",
    screening_data[
        "P_matrix_question"
    ].nunique()
)

Screening data shape: (63328, 57)
Unique questions: 53


In [52]:
P_feature_cols = [
    f"P_{i}{j}"
    for i in range(1, 5)
    for j in range(1, 5)
]

print(
    "P columns found:",
    len(
        [
            c
            for c in P_feature_cols
            if c in screening_data.columns
        ]
    )
)

P columns found: 16


In [54]:
unique_P_reference = (
    screening_data[
        [
            "P_matrix_question"
        ]
        + P_feature_cols
    ]
    .drop_duplicates(
        subset=P_feature_cols
    )
    .reset_index(
        drop=True
    )
)

print(
    "Unique P structures represented:",
    len(
        unique_P_reference
    )
)

print(
    "Question labels represented:",
    unique_P_reference[
        "P_matrix_question"
    ].nunique()
)

Unique P structures represented: 63328
Question labels represented: 53


In [56]:
def calculate_P_structure_descriptors(
    P_matrix
):
    P = np.asarray(
        P_matrix,
        dtype=float
    )

    diag = np.diag(
        P
    )

    row_sums = P.sum(
        axis=1
    )

    return {
        "certainty_P":
            rya_certainty(P),

        "rya_asymmetry_P":
            rya_asymmetry(P),

        "min_diag_P":
            float(
                diag.min()
            ),

        "max_diag_P":
            float(
                diag.max()
            ),

        "mean_diag_P":
            float(
                diag.mean()
            ),

        "max_row_sum_error_P":
            float(
                np.max(
                    np.abs(
                        row_sums - 1.0
                    )
                )
            ),

        "all_rows_diagonal_mode_P":
            int(
                np.all(
                    diag
                    >= P.max(
                        axis=1
                    )
                )
            )
    }

In [58]:
P_structure_rows = []

for _, row in unique_P_reference.iterrows():

    P = np.array([
        [
            row["P_11"],
            row["P_12"],
            row["P_13"],
            row["P_14"]
        ],
        [
            row["P_21"],
            row["P_22"],
            row["P_23"],
            row["P_24"]
        ],
        [
            row["P_31"],
            row["P_32"],
            row["P_33"],
            row["P_34"]
        ],
        [
            row["P_41"],
            row["P_42"],
            row["P_43"],
            row["P_44"]
        ]
    ], dtype=float)

    descriptors = (
        calculate_P_structure_descriptors(
            P
        )
    )

    descriptors[
        "P_matrix_question"
    ] = row[
        "P_matrix_question"
    ]

    P_structure_rows.append(
        descriptors
    )


P_structure_reference = pd.DataFrame(
    P_structure_rows
)

print(
    "Reference descriptor table:",
    P_structure_reference.shape
)

display(
    P_structure_reference.head()
)

Reference descriptor table: (63328, 8)


,certainty_P,rya_asymmetry_P,min_diag_P,max_diag_P,mean_diag_P,max_row_sum_error_P,all_rows_diagonal_mode_P,P_matrix_question
0,3.068,0.005607,0.635,0.875,0.76700,1.110223e-16,1,cyb1
1,1.887,0.026359,0.328,0.666,0.47175,0.000000e+00,0,cop1
2,2.633,0.003265,0.526,0.737,0.65825,0.000000e+00,1,phq5
3,3.012,0.001966,0.698,0.841,0.75300,1.000000e-03,1,ry19
4,2.402,0.029987,0.432,0.801,0.60050,0.000000e+00,1,ry9


In [60]:
descriptor_cols = [
    "certainty_P",
    "rya_asymmetry_P",
    "min_diag_P",
    "max_diag_P",
    "mean_diag_P",
    "max_row_sum_error_P"
]

training_space_summary = (
    P_structure_reference[
        descriptor_cols
    ]
    .describe()
    .T
)

display(
    training_space_summary
)

,count,mean,std,min,25%,50%,75%,max
certainty_P,63328.0,2.525198,0.210532,1.848000,2.387000,2.537000,2.63700,3.109000
rya_asymmetry_P,63328.0,0.014256,0.009474,0.000306,0.007221,0.012182,0.01889,0.049194
min_diag_P,63328.0,0.516971,0.077278,0.316000,0.466000,0.507000,0.57700,0.701000
max_diag_P,63328.0,0.776882,0.078081,0.581000,0.723000,0.780000,0.82900,0.968000
mean_diag_P,63328.0,0.631300,0.052633,0.462000,0.596750,0.634250,0.65925,0.777250
max_row_sum_error_P,63328.0,0.000803,0.000398,0.000000,0.001000,0.001000,0.00100,0.001000


In [62]:
ry9_P_descriptors = (
    calculate_P_structure_descriptors(
        P_ry9
    )
)

ry9_P_descriptors

{'certainty_P': 2.42,
 'rya_asymmetry_P': 0.030716666666666666,
 'min_diag_P': 0.43,
 'max_diag_P': 0.81,
 'mean_diag_P': 0.605,
 'max_row_sum_error_P': 0.010000000000000009,
 'all_rows_diagonal_mode_P': 1}

In [64]:
ry9_structure_comparison = []

for feature in descriptor_cols:

    value = float(
        ry9_P_descriptors[
            feature
        ]
    )

    training_min = float(
        P_structure_reference[
            feature
        ].min()
    )

    training_max = float(
        P_structure_reference[
            feature
        ].max()
    )

    within_range = (
        training_min
        <= value
        <= training_max
    )

    ry9_structure_comparison.append({
        "descriptor":
            feature,

        "value":
            value,

        "training_min":
            training_min,

        "training_max":
            training_max,

        "within_observed_range":
            within_range
    })


ry9_structure_comparison_df = pd.DataFrame(
    ry9_structure_comparison
)

display(
    ry9_structure_comparison_df
)

,descriptor,value,training_min,training_max,within_observed_range
0,certainty_P,2.420000,1.848000,3.109000,True
1,rya_asymmetry_P,0.030717,0.000306,0.049194,True
2,min_diag_P,0.430000,0.316000,0.701000,True
3,max_diag_P,0.810000,0.581000,0.968000,True
4,mean_diag_P,0.605000,0.462000,0.777250,True
5,max_row_sum_error_P,0.010000,0.000000,0.001000,False


### P-matrix training-space finding

The 63,328 historical RYA instances contain 53 question labels but 63,328
distinct 4x4 P-matrix structures.

This indicates that the detector-development data contains instance-specific
matrix variation rather than a single fixed matrix for each question.

For the live `ry9` matrix, all principal structural descriptors fall within
the range represented by the historical instance data:

- certainty;
- RYA asymmetry;
- minimum diagonal;
- maximum diagonal;
- mean diagonal.

The only out-of-range quantity is maximum row-sum error. The live `ry9`
matrix has a maximum deviation of 0.01, whereas the historical instance
matrices reach approximately 0.001.

Row-sum error is treated as a mathematical validity / rounding diagnostic,
not as a structural out-of-distribution descriptor. The matrix is therefore
retained with a warning rather than automatically routed to fallback.

In [67]:
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors

P_training_X = (
    screening_data[
        P_feature_cols
    ]
    .astype(float)
    .copy()
)

print(
    "P training matrix:",
    P_training_X.shape
)

P training matrix: (63328, 16)


In [69]:
P_space_scaler = StandardScaler()

P_training_scaled = (
    P_space_scaler.fit_transform(
        P_training_X
    )
)

print(
    "Scaled P space:",
    P_training_scaled.shape
)

Scaled P space: (63328, 16)


In [71]:
ry9_P_vector = np.array(
    [
        P_ry9[i, j]
        for i in range(4)
        for j in range(4)
    ],
    dtype=float
).reshape(1, -1)

ry9_P_scaled = (
    P_space_scaler.transform(
        ry9_P_vector
    )
)

print(
    "ry9 vector shape:",
    ry9_P_scaled.shape
)

ry9 vector shape: (1, 16)


C:\Users\Willy\anaconda3\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [73]:
ry9_P_df = pd.DataFrame(
    ry9_P_vector,
    columns=P_feature_cols
)

ry9_P_scaled = (
    P_space_scaler.transform(
        ry9_P_df
    )
)

In [75]:
P_nn = NearestNeighbors(
    n_neighbors=1,
    metric="euclidean"
)

P_nn.fit(
    P_training_scaled
)

ry9_distance, ry9_index = (
    P_nn.kneighbors(
        ry9_P_scaled
    )
)

ry9_nearest_distance = float(
    ry9_distance[0, 0]
)

ry9_nearest_index = int(
    ry9_index[0, 0]
)

print(
    "ry9 nearest training-space distance:",
    ry9_nearest_distance
)

print(
    "Nearest historical row index:",
    ry9_nearest_index
)

print(
    "Nearest question:",
    screening_data.iloc[
        ry9_nearest_index
    ][
        "P_matrix_question"
    ]
)

ry9 nearest training-space distance: 0.16115591409120023
Nearest historical row index: 13716
Nearest question: ry9


C:\Users\Willy\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] 系統找不到指定的檔案。
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "C:\Users\Willy\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "C:\Users\Willy\anaconda3\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Willy\anaconda3\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "C:\Users\Willy\anaconda3\Lib\subprocess.py", line 1538, in _execute_child
    hp, ht, pid, tid = _winapi.CreateProcess(executable, args,
       

In [77]:
rng = np.random.default_rng(
    2026
)

reference_indices = rng.choice(
    len(
        P_training_scaled
    ),
    size=5000,
    replace=False
)

P_reference_sample = (
    P_training_scaled[
        reference_indices
    ]
)

reference_nn = NearestNeighbors(
    n_neighbors=2,
    metric="euclidean"
)

reference_nn.fit(
    P_training_scaled
)

reference_distances, _ = (
    reference_nn.kneighbors(
        P_reference_sample
    )
)

# First neighbour is the case itself.
# Second neighbour is the nearest other training matrix.
training_nearest_distances = (
    reference_distances[
        :, 1
    ]
)

print(
    "Training nearest-distance median:",
    np.median(
        training_nearest_distances
    )
)

print(
    "Training nearest-distance 95th percentile:",
    np.quantile(
        training_nearest_distances,
        0.95
    )
)

print(
    "Training nearest-distance 99th percentile:",
    np.quantile(
        training_nearest_distances,
        0.99
    )
)

print(
    "ry9 distance:",
    ry9_nearest_distance
)

Training nearest-distance median: 0.24927934752859998
Training nearest-distance 95th percentile: 0.3471877416722292
Training nearest-distance 99th percentile: 0.39935610897112467
ry9 distance: 0.16115591409120023


In [79]:
ry9_distance_percentile = float(
    np.mean(
        training_nearest_distances
        <= ry9_nearest_distance
    )
    * 100
)

print(
    "ry9 empirical distance percentile:",
    ry9_distance_percentile
)

ry9 empirical distance percentile: 1.46


### Training-space distance result for `ry9`

The live `ry9` unreliability matrix is well represented by the historical
P-matrix training space.

Its nearest-neighbour distance in the standardised 16-dimensional P-matrix
space was 0.1612.

For comparison, historical training matrices had:

- median nearest-neighbour distance = 0.2493;
- 95th percentile = 0.3472;
- 99th percentile = 0.3994.

The `ry9` matrix was at approximately the 1.46th empirical percentile of
nearest-neighbour distance.

Because lower distance indicates greater similarity to a represented training
matrix, this places `ry9` comfortably inside the observed P-matrix training
space.

This diagnostic is currently used as evidence of training-space similarity
rather than as a hard fallback rule.

## 3.4 Compare a known incompatible case with P-matrix training space

A previously identified high-confidence incompatible case is examined using
the same P-matrix training-space distance diagnostic.

The case:

- exp_code: `orthog_B_3_large_classes_2166`
- question: `ry24`
- n_X = 6
- n_Y = 5215
- compatibility-detector probability ≈ 0.9999999

This test helps distinguish whether incompatibility is associated with an
unusual unreliability matrix, or whether it is mainly driven by other factors
such as extreme respondent-group-size imbalance.

In [83]:
target_exp_code = "orthog_B_3_large_classes_2166"

ry24_case = (
    screening_data[
        screening_data["exp_code"]
        == target_exp_code
    ]
)

print(
    "Matching rows:",
    len(ry24_case)
)

if len(ry24_case) != 1:
    raise ValueError(
        f"Expected exactly one row for {target_exp_code}"
    )

ry24_case = ry24_case.iloc[0]

print(
    "Question:",
    ry24_case["P_matrix_question"]
)

print(
    "n_X:",
    ry24_case["n_X"]
)

print(
    "n_Y:",
    ry24_case["n_Y"]
)

print(
    "Group-size ratio:",
    max(
        ry24_case["n_X"],
        ry24_case["n_Y"]
    )
    /
    min(
        ry24_case["n_X"],
        ry24_case["n_Y"]
    )
)

Matching rows: 1
Question: ry24
n_X: 6
n_Y: 5215
Group-size ratio: 869.1666666666666


In [85]:
P_ry24_instance = np.array([
    [
        ry24_case["P_11"],
        ry24_case["P_12"],
        ry24_case["P_13"],
        ry24_case["P_14"]
    ],
    [
        ry24_case["P_21"],
        ry24_case["P_22"],
        ry24_case["P_23"],
        ry24_case["P_24"]
    ],
    [
        ry24_case["P_31"],
        ry24_case["P_32"],
        ry24_case["P_33"],
        ry24_case["P_34"]
    ],
    [
        ry24_case["P_41"],
        ry24_case["P_42"],
        ry24_case["P_43"],
        ry24_case["P_44"]
    ]
], dtype=float)

print(
    "ry24 instance-specific P matrix:"
)

print(
    P_ry24_instance
)

print(
    "\nRow sums:"
)

print(
    P_ry24_instance.sum(
        axis=1
    )
)

ry24 instance-specific P matrix:
[[0.463 0.262 0.166 0.109]
 [0.063 0.495 0.293 0.149]
 [0.009 0.131 0.563 0.298]
 [0.017 0.007 0.141 0.835]]

Row sums:
[1.    1.    1.001 1.   ]


In [87]:
ry24_P_descriptors = (
    calculate_P_structure_descriptors(
        P_ry24_instance
    )
)

ry24_P_descriptors

{'certainty_P': 2.356,
 'rya_asymmetry_P': 0.02396183333333333,
 'min_diag_P': 0.463,
 'max_diag_P': 0.835,
 'mean_diag_P': 0.589,
 'max_row_sum_error_P': 0.0009999999999998899,
 'all_rows_diagonal_mode_P': 1}

In [89]:
ry24_structure_comparison = []

for feature in descriptor_cols:

    value = float(
        ry24_P_descriptors[
            feature
        ]
    )

    training_min = float(
        P_structure_reference[
            feature
        ].min()
    )

    training_max = float(
        P_structure_reference[
            feature
        ].max()
    )

    within_range = (
        training_min
        <= value
        <= training_max
    )

    ry24_structure_comparison.append({
        "descriptor":
            feature,

        "value":
            value,

        "training_min":
            training_min,

        "training_max":
            training_max,

        "within_observed_range":
            within_range
    })


ry24_structure_comparison_df = (
    pd.DataFrame(
        ry24_structure_comparison
    )
)

display(
    ry24_structure_comparison_df
)

,descriptor,value,training_min,training_max,within_observed_range
0,certainty_P,2.356000,1.848000,3.109000,True
1,rya_asymmetry_P,0.023962,0.000306,0.049194,True
2,min_diag_P,0.463000,0.316000,0.701000,True
3,max_diag_P,0.835000,0.581000,0.968000,True
4,mean_diag_P,0.589000,0.462000,0.777250,True
5,max_row_sum_error_P,0.001000,0.000000,0.001000,True


In [91]:
ry24_P_vector = np.array(
    [
        P_ry24_instance[i, j]
        for i in range(4)
        for j in range(4)
    ],
    dtype=float
).reshape(1, -1)

ry24_P_df = pd.DataFrame(
    ry24_P_vector,
    columns=P_feature_cols
)

ry24_P_scaled = (
    P_space_scaler.transform(
        ry24_P_df
    )
)

ry24_distance, ry24_index = (
    P_nn.kneighbors(
        ry24_P_scaled
    )
)

ry24_nearest_distance = float(
    ry24_distance[0, 0]
)

ry24_nearest_index = int(
    ry24_index[0, 0]
)

print(
    "ry24 nearest training-space distance:",
    ry24_nearest_distance
)

print(
    "Nearest historical row index:",
    ry24_nearest_index
)

print(
    "Nearest question:",
    screening_data.iloc[
        ry24_nearest_index
    ][
        "P_matrix_question"
    ]
)

ry24 nearest training-space distance: 0.0
Nearest historical row index: 2165
Nearest question: ry24


In [93]:
ry24_distances, ry24_indices = (
    P_nn.kneighbors(
        ry24_P_scaled,
        n_neighbors=2
    )
)

print(
    "Nearest neighbour distance:",
    ry24_distances[0, 0]
)

print(
    "Second-nearest distance:",
    ry24_distances[0, 1]
)

print(
    "Nearest row:",
    ry24_indices[0, 0]
)

print(
    "Second-nearest row:",
    ry24_indices[0, 1]
)

Nearest neighbour distance: 0.0
Second-nearest distance: 0.24665653739082505
Nearest row: 2165
Second-nearest row: 36994


In [95]:
ry24_nearest_other_distance = float(
    ry24_distances[0, 1]
)

print(
    "ry24 nearest-other training distance:",
    ry24_nearest_other_distance
)

ry24 nearest-other training distance: 0.24665653739082505


In [97]:
ry24_distance_percentile = float(
    np.mean(
        training_nearest_distances
        <= ry24_nearest_other_distance
    )
    * 100
)

print(
    "ry24 empirical distance percentile:",
    ry24_distance_percentile
)

ry24 empirical distance percentile: 47.68


In [99]:
P_distance_comparison = pd.DataFrame({
    "case": [
        "ry9 live matrix",
        "ry24 incompatible instance"
    ],

    "distance": [
        ry9_nearest_distance,
        ry24_nearest_other_distance
    ],

    "empirical_percentile": [
        ry9_distance_percentile,
        ry24_distance_percentile
    ]
})

display(
    P_distance_comparison
)

,case,distance,empirical_percentile
0,ry9 live matrix,0.161156,1.46
1,ry24 incompatible instance,0.246657,47.68


### Known incompatible case: training-space interpretation

The high-confidence incompatible `ry24` case was also evaluated against the
historical P-matrix training space.

All matrix structural descriptors were within the observed historical ranges.

Its nearest-other distance in the standardised 16-dimensional P-matrix space
was 0.2467, corresponding to approximately the 47.68th percentile of the
historical nearest-neighbour distance distribution.

This indicates that the unreliability matrix itself is not unusual relative
to the represented training space.

Despite this, the compatibility classifier assigns the case a near-certain
incompatibility probability. The case has group sizes of 6 and 5215, producing
an extreme imbalance ratio of approximately 869.

This supports the design of Objective 1 as two distinct checks:

1. matrix validity / training-space compatibility; and
2. Gamma compatibility of the full comparison.

A matrix can be structurally ordinary while the overall comparison remains
unsuitable for the Gamma neural-network pathway.

## 3.5 Synthetic out-of-distribution P-matrix stress test

A deliberately unusual 4x4 row-stochastic matrix is used to test whether the
training-space diagnostic can identify a structure that is far from the
historical RYA P-matrix space.

This matrix is not proposed as a realistic RYA unreliability matrix. It is
used only as a diagnostic stress test.

The matrix remains mathematically valid:

- all entries are non-negative;
- each row sums to 1.

However, its structure is intentionally very different from the historically
observed matrices.

In [103]:
P_synthetic_ood = np.array([
    [0.10, 0.70, 0.10, 0.10],
    [0.65, 0.10, 0.15, 0.10],
    [0.10, 0.15, 0.10, 0.65],
    [0.10, 0.10, 0.70, 0.10]
], dtype=float)

print("Synthetic OOD P matrix:")
print(P_synthetic_ood)

print("\nRow sums:")
print(
    P_synthetic_ood.sum(axis=1)
)

print(
    "\nNon-negative:",
    np.all(
        P_synthetic_ood >= 0
    )
)

Synthetic OOD P matrix:
[[0.1  0.7  0.1  0.1 ]
 [0.65 0.1  0.15 0.1 ]
 [0.1  0.15 0.1  0.65]
 [0.1  0.1  0.7  0.1 ]]

Row sums:
[1. 1. 1. 1.]

Non-negative: True


In [105]:
synthetic_ood_descriptors = (
    calculate_P_structure_descriptors(
        P_synthetic_ood
    )
)

synthetic_ood_descriptors

{'certainty_P': 0.4,
 'rya_asymmetry_P': 0.0008333333333333312,
 'min_diag_P': 0.1,
 'max_diag_P': 0.1,
 'mean_diag_P': 0.1,
 'max_row_sum_error_P': 1.1102230246251565e-16,
 'all_rows_diagonal_mode_P': 0}

In [107]:
synthetic_structure_comparison = []

for feature in descriptor_cols:

    value = float(
        synthetic_ood_descriptors[
            feature
        ]
    )

    training_min = float(
        P_structure_reference[
            feature
        ].min()
    )

    training_max = float(
        P_structure_reference[
            feature
        ].max()
    )

    within_range = (
        training_min
        <= value
        <= training_max
    )

    synthetic_structure_comparison.append({
        "descriptor":
            feature,

        "value":
            value,

        "training_min":
            training_min,

        "training_max":
            training_max,

        "within_observed_range":
            within_range
    })


synthetic_structure_comparison_df = pd.DataFrame(
    synthetic_structure_comparison
)

display(
    synthetic_structure_comparison_df
)

,descriptor,value,training_min,training_max,within_observed_range
0,certainty_P,4.000000e-01,1.848000,3.109000,False
1,rya_asymmetry_P,8.333333e-04,0.000306,0.049194,True
2,min_diag_P,1.000000e-01,0.316000,0.701000,False
3,max_diag_P,1.000000e-01,0.581000,0.968000,False
4,mean_diag_P,1.000000e-01,0.462000,0.777250,False
5,max_row_sum_error_P,1.110223e-16,0.000000,0.001000,True


In [109]:
synthetic_P_vector = np.array(
    [
        P_synthetic_ood[i, j]
        for i in range(4)
        for j in range(4)
    ],
    dtype=float
).reshape(1, -1)

synthetic_P_df = pd.DataFrame(
    synthetic_P_vector,
    columns=P_feature_cols
)

synthetic_P_scaled = (
    P_space_scaler.transform(
        synthetic_P_df
    )
)

synthetic_distance, synthetic_index = (
    P_nn.kneighbors(
        synthetic_P_scaled
    )
)

synthetic_nearest_distance = float(
    synthetic_distance[0, 0]
)

synthetic_nearest_index = int(
    synthetic_index[0, 0]
)

print(
    "Synthetic OOD nearest training-space distance:",
    synthetic_nearest_distance
)

print(
    "Nearest historical row index:",
    synthetic_nearest_index
)

print(
    "Nearest question:",
    screening_data.iloc[
        synthetic_nearest_index
    ][
        "P_matrix_question"
    ]
)

Synthetic OOD nearest training-space distance: 21.11760125857671
Nearest historical row index: 28668
Nearest question: ph2


In [111]:
synthetic_distance_percentile = float(
    np.mean(
        training_nearest_distances
        <= synthetic_nearest_distance
    )
    * 100
)

print(
    "Synthetic OOD empirical distance percentile:",
    synthetic_distance_percentile
)

Synthetic OOD empirical distance percentile: 100.0


In [113]:
P_distance_comparison_final = pd.DataFrame({
    "case": [
        "ry9 live matrix",
        "ry24 incompatible instance",
        "synthetic OOD matrix"
    ],

    "distance": [
        ry9_nearest_distance,
        ry24_nearest_other_distance,
        synthetic_nearest_distance
    ],

    "empirical_percentile": [
        ry9_distance_percentile,
        ry24_distance_percentile,
        synthetic_distance_percentile
    ]
})

display(
    P_distance_comparison_final
)

,case,distance,empirical_percentile
0,ry9 live matrix,0.161156,1.46
1,ry24 incompatible instance,0.246657,47.68
2,synthetic OOD matrix,21.117601,100.00


## 4. O1-08 conclusion

The remaining Objective 1 training-space validation requirement was addressed
successfully.

Three representative cases were evaluated.

### `ry9` live matrix

The `ry9` matrix was strongly represented in historical P-matrix space.

- nearest training-space distance = 0.1612;
- empirical percentile = 1.46%;
- principal structural descriptors were within the historical ranges.

### Known incompatible `ry24` case

The high-confidence incompatible `ry24` case also had an ordinary unreliability
matrix.

- nearest-other training-space distance = 0.2467;
- empirical percentile = 47.68%;
- all principal matrix descriptors were within historical ranges.

This shows that Gamma incompatibility does not necessarily imply that the
unreliability matrix itself is unusual. In this case, the extreme respondent
group-size imbalance is a more plausible driver of incompatibility.

### Synthetic out-of-distribution matrix

A deliberately unusual but mathematically valid row-stochastic matrix was used
as a stress test.

The synthetic matrix had:

- certainty = 0.40, below the historical range;
- minimum, maximum and mean diagonal values below the historical ranges;
- nearest training-space distance = 21.1176;
- empirical distance percentile = 100%.

The diagnostic therefore clearly identified the synthetic matrix as structurally
unlike matrices represented in the historical RYA instance data.

### Interpretation

The Objective 1 gate now contains two complementary forms of screening:

1. unreliability-matrix validity and training-space compatibility; and
2. full-comparison Gamma compatibility classification.

These checks address different failure mechanisms and should remain separate.

Training-space distance is currently treated as a diagnostic rather than a hard
routing threshold. A production cutoff should not be introduced without further
validation or agreement with RYA.